<a href="https://colab.research.google.com/github/botirbekzulf-sudo/ML/blob/main/%D0%A3%D1%80%D0%BE%D0%BA18_LLM_%D0%BF%D1%80%D0%B0%D0%BA%D1%82%D0%B8%D0%BA%D0%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 💬 Урок 18 — LLM на практике: анализ отзывов и свой бот

Решаем НАСТОЯЩУЮ задачу: автоматически определяем тональность отзывов и делаем бота с характером.

> План: 1) готовая модель тональности → 2) русские отзывы → 3) где ломается наивный метод → 4) промпт (роль+примеры+формат) → 5) бот с характером → ⭐ мини-RAG → ⭐⭐ путь через API.

**Ядро урока работает БЕЗ API-ключей** (модели Hugging Face скачиваются бесплатно). Ключ нужен только в самом конце — необязательный шаг ⭐⭐.

## ⚠️ Два разных `pipeline` / `Pipeline` — не путай!

| Что | Откуда | Зачем |
|---|---|---|
| `Pipeline` (с большой буквы) | `from sklearn.pipeline import Pipeline` | склеить СВОЮ предобработку + модель (прошлые уроки) |
| `pipeline` (с маленькой) | `from transformers import pipeline` | загрузить ЧУЖУЮ готовую модель одной строкой |

Названия похожи — инструменты разные.

## Шаг 1 · Готовая модель тональности (Hugging Face, без ключа)
Одна строка — и модель, обученная различать позитив/негатив. Ничего обучать не надо. *(нужен интернет — как в Colab)*

In [1]:
!pip install transformers -q
from transformers import pipeline

# англоязычная модель (быстрая, для демонстрации)
sentiment = pipeline('sentiment-analysis',
                     model='distilbert-base-uncased-finetuned-sst-2-english')

print(sentiment('This movie was absolutely wonderful, I loved it!'))
# вернёт label (POSITIVE/NEGATIVE) и score — уверенность модели от 0 до 1

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

[{'label': 'POSITIVE', 'score': 0.9998749494552612}]


**❓ Вопрос.** Модель вернула `label` и `score`. `score = 0.99` — это что? А если `score = 0.55` — можно ли доверять метке? *(ответ: 0.55 — модель почти не уверена, проверяй вручную)*

## Шаг 2 · Русские отзывы → многоязычная модель
Англоязычная модель плохо понимает русский. Берём многоязычную — она отвечает в «звёздах» (1–5).

In [24]:
multi = pipeline('sentiment-analysis',
                 model='nlptown/bert-base-multilingual-uncased-sentiment')

мои_отзывы = [
    'Эта игра затягивает на часы, обожаю её!',   # 👈 замени на свои
    'Ужасный сервис, больше не приду.',
    'Неплохо, но дороговато.',
    'Лучшее кафе в городе!',
    '戴蒙',
    'Жақсы',
]
поз = нег = 0
for о in мои_отзывы:
    звёзд = int(multi(о)[0]['label'][0])            # '4 stars' -> 4
    вердикт = 'ПОЗИТИВ' if звёзд>=4 else ('НЕГАТИВ' if звёзд<=2 else 'НЕЙТРАЛ')
    print(f'{вердикт:8} {звёзд}★  <-  {о}')
    поз += звёзд>=4; нег += звёзд<=2
print(f'\nИтого: позитивных {поз}, негативных {нег}')

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

ПОЗИТИВ  5★  <-  Эта игра затягивает на часы, обожаю её!
НЕГАТИВ  1★  <-  Ужасный сервис, больше не приду.
НЕЙТРАЛ  3★  <-  Неплохо, но дороговато.
ПОЗИТИВ  5★  <-  Лучшее кафе в городе!
ПОЗИТИВ  4★  <-  戴蒙
ПОЗИТИВ  5★  <-  Жақсы

Итого: позитивных 4, негативных 1


## Шаг 3 · Наивный словарь — и где он ломается
Простейший метод: считаем позитивные и негативные слова. Работает офлайн — и хорошо показывает СЛАБЫЕ места.

In [19]:
позитив = {'отличный','супер','советую','нравится','быстро','вежливо','класс','порадовал'}
негатив = {'ужасно','плохо','долго','грубо','обман','отвратительно'}

def словарь(текст):
    слова = текст.lower().replace('!','').replace('.','').replace(',','').split()
    p = sum(w in позитив for w in слова)
    n = sum(w in негатив for w in слова)
    return 'позитив' if p>n else 'негатив' if n>p else 'нейтрально'

тесты = [
    ('отличный сервис, всем советую',  'позитив'),
    ('ужасно и долго',                 'негатив'),
    ('ну очень порадовал этот сервис', 'НЕГАТИВ (сарказм!)'),
    ('не плохо',                       'ПОЗИТИВ (отрицание!)'),
]
for текст, правда in тесты:
    print(f'словарь: {словарь(текст):11} | на деле: {правда:22} | {текст}')

словарь: позитив     | на деле: позитив                | отличный сервис, всем советую
словарь: негатив     | на деле: негатив                | ужасно и долго
словарь: позитив     | на деле: НЕГАТИВ (сарказм!)     | ну очень порадовал этот сервис
словарь: негатив     | на деле: ПОЗИТИВ (отрицание!)   | не плохо


✍️ **Ответь.** На каких двух отзывах словарь ошибся и почему? *(подсказка: сарказм и слово «не» перед оценкой — словарь смотрит на отдельные слова, а не на смысл)*

## Шаг 4 · Хороший промпт: роль + примеры + формат
Промпт — это задание модели. Три приёма: **роль**, **примеры** (few-shot), **формат** ответа.
Соберём промпт для тональности и вставим его в чат-LLM (ChatGPT / Claude).

In [20]:
отзыв = 'Ну очень порадовал этот сервис...'   # 👈 меняй

промпт = f'''Ты анализируешь отзывы (роль).
Определи тональность и ответь ОДНИМ словом: позитив, негатив или нейтрально (формат).

Примеры:
«Всё быстро и вежливо» -> позитив
«Ждал час, никто не помог» -> негатив

Отзыв: «{отзыв}» ->'''

print(промпт)
# Скопируй этот промпт в ChatGPT/Claude. Умная модель поймёт сарказм там, где словарь ошибся.

Ты анализируешь отзывы (роль).
Определи тональность и ответь ОДНИМ словом: позитив, негатив или нейтрально (формат).

Примеры:
«Всё быстро и вежливо» -> позитив
«Ждал час, никто не помог» -> негатив

Отзыв: «Ну очень порадовал этот сервис...» ->


### 📝 Задание (базовый)
Собери 8 своих отзывов, прогони через модель из шага 2, найди отзыв, где модель ошиблась, и объясни почему.

### 📝 Задание (продвинутый)
Напиши свой few-shot промпт для другой задачи (сортировка писем, помощник по учёбе) и проверь его в чат-LLM.

## Шаг 5 · Бот с характером (системный промпт)
**Системный промпт** — постоянная инструкция «кто ты», действует на весь разговор. Меняешь одну строку — меняется вся личность бота.

In [21]:
system = 'Ты — дружелюбный кот-программист. Отвечай коротко, с юмором.'  # 👈 придумай свою роль
вопрос = 'Что такое переменная?'

print('SYSTEM:', system)
print('USER  :', вопрос)
print('\nВставь SYSTEM как «системный промпт», а вопрос — как сообщение, в ChatGPT/Claude.')
# Задание: придумай 3 разные роли (пират, строгий учитель, робот) и сравни ответы на один вопрос.

SYSTEM: Ты — дружелюбный кот-программист. Отвечай коротко, с юмором.
USER  : Что такое переменная?

Вставь SYSTEM как «системный промпт», а вопрос — как сообщение, в ChatGPT/Claude.


## ⭐ Шаг 6 · Мини-RAG: ответ строго по своему тексту
RAG-lite уменьшает галлюцинации: даём модели проверенный текст и просим отвечать ТОЛЬКО по нему. Здесь — простой строковый поиск нужного куска (работает офлайн и на любом языке).

In [22]:
текст = '''Пингвины Палмера живут в Антарктике. Есть три вида: Adelie, Chinstrap и Gentoo.
Gentoo — самые крупные из трёх.'''

вопрос = 'Какой вид пингвинов самый крупный?'

# для короткого текста отдаём модели ВЕСЬ контекст и просим отвечать только по нему
промпт = f'''Ответь на вопрос ТОЛЬКО по тексту ниже. Если ответа в тексте нет — скажи «не знаю».

Текст: {текст}

Вопрос: {вопрос}'''
print(промпт)
# Вставь в чат-LLM. Потом задай вопрос, которого в тексте НЕТ, — хорошая модель ответит «не знаю».

Ответь на вопрос ТОЛЬКО по тексту ниже. Если ответа в тексте нет — скажи «не знаю».

Текст: Пингвины Палмера живут в Антарктике. Есть три вида: Adelie, Chinstrap и Gentoo.
Gentoo — самые крупные из трёх.

Вопрос: Какой вид пингвинов самый крупный?


## ⭐⭐ (необязательно) Путь через API — если у тебя есть ключ
Всё выше работало без ключей. Этот блок — для тех, кто хочет вызывать модель прямо из кода. Ключ храни в «Секретах» Colab (значок 🔑 слева), а НЕ в коде: ключ = пароль, за утечку платит владелец.

In [23]:
!pip install anthropic -q
import anthropic
from google.colab import userdata   # доступ к «Секретам» Colab

client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))

def спроси(промпт):
    msg = client.messages.create(
        model='claude-haiku-4-5',      # имена версий меняются — сверься с docs.claude.com
        max_tokens=50,
        messages=[{'role':'user','content':промпт}],
    )
    return msg.content[0].text          # достаём текст из ответа

print(спроси('Определи тональность одним словом: «Ну очень порадовал этот сервис...»'))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 17.4 MB/s eta 0:00:00


SecretNotFoundError: Secret ANTHROPIC_API_KEY does not exist.

## Мини-итог

- HF `pipeline` (маленькая буква) — загрузчик чужой готовой модели; sklearn `Pipeline` (большая) — твой конвейер предобработки+модели.
- Три приёма промпта: **роль**, **примеры**, **формат** (+ бонус «рассуждай по шагам»).
- Наивный словарь ломается на **сарказме** и **отрицании** — умная модель справляется, но её ответ всё равно проверяем.
- **RAG** уменьшает галлюцинации: модель отвечает по проверенному тексту.

> Ты решил реальную задачу — автоматический разбор отзывов — на бесплатных моделях. Это не «ещё один чат-бот».